# 🏅 Medallion Pipeline v2 — HR Employee Attrition
**Stack:** PySpark + MinIO (S3-compatible)

```
CSV (local Jupyter)
    │
    ▼
datalake/                          ← 1 bucket MinIO
├── raw/hr-attrition/              🥉 BRONZE  → raw parquet, no transform
└── processed/
    ├── silver/hr-attrition/       🥈 SILVER  → cleaning, cast, encode
    └── gold/hr-attrition/         🥇 GOLD    → ML-ready features
```

## 0️⃣ Setup — Konfigurasi & SparkSession

In [1]:
# ── Konfigurasi MinIO ──────────────────────────────────────────────────────
MINIO_ENDPOINT   = "http://minio-kelompok3:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"

# Semua layer dalam satu bucket 'datalake'
BUCKET      = "datalake"
BRONZE_PATH = "s3a://datalake/raw/hr-attrition/"
SILVER_PATH = "s3a://datalake/processed/silver/hr-attrition/"
GOLD_PATH   = "s3a://datalake/processed/gold/hr-attrition/"

CSV_SOURCE = "/home/jovyan/work/HR-Employee-Attrition.csv"
print("✅ Konfigurasi selesai")

✅ Konfigurasi selesai


In [2]:
# Install dependency
import subprocess
subprocess.run(['pip', 'install', 'boto3', '-q'], check=True)
print("✅ boto3 siap")

✅ boto3 siap


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

spark = (
    SparkSession.builder
    .appName("HR-Attrition-Medallion-v2")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.endpoint",               MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",             MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key",             MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access",      "true")
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.sql.shuffle.partitions",               "4")
    .config("spark.sql.parquet.compression.codec",        "snappy")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("✅ SparkSession aktif:", spark.version)

✅ SparkSession aktif: 3.5.0


In [4]:
# ── Verifikasi bucket 'datalake' tersedia ────────────────────────────────
import boto3
from botocore.client import Config

s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version="s3v4"),
    region_name="us-east-1"
)

existing = [b["Name"] for b in s3.list_buckets()["Buckets"]]
if BUCKET in existing:
    print(f"✅ Bucket '{BUCKET}' ditemukan")
    # Tampilkan folder yang sudah ada
    resp = s3.list_objects_v2(Bucket=BUCKET, Delimiter='/')
    folders = [p['Prefix'] for p in resp.get('CommonPrefixes', [])]
    print(f"   Folder existing: {folders}")
else:
    raise Exception(f"❌ Bucket '{BUCKET}' tidak ditemukan! Pastikan sudah dibuat di MinIO.")

✅ Bucket 'datalake' ditemukan
   Folder existing: []


---
## 🥉 BRONZE — Raw Ingestion
> Baca CSV apa adanya → simpan Parquet ke MinIO. **Zero transform.**

In [5]:
from datetime import datetime

df_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(CSV_SOURCE)
)

# Metadata ingestion
df_bronze = df_bronze \
    .withColumn("_ingested_at", F.lit(datetime.utcnow().isoformat())) \
    .withColumn("_source_file", F.lit("HR-Employee-Attrition.csv")) \
    .withColumn("_layer",       F.lit("bronze"))

print(f"📦 Bronze — {df_bronze.count()} baris, {len(df_bronze.columns)} kolom")

(
    df_bronze
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(BRONZE_PATH)
)
print(f"✅ Bronze → {BRONZE_PATH}")

📦 Bronze — 1476 baris, 38 kolom
✅ Bronze → s3a://datalake/raw/hr-attrition/


---
## 🥈 SILVER — Data Cleaning
> Baca Bronze → bersihkan data → simpan ke MinIO.

| Step | Aksi |
|------|------|
| 1 | Fix BOM character di nama kolom |
| 2 | Rename `Attrition` → `left_company`, encode Yes/No → 1/0 |
| 3 | Drop kolom tidak informatif |
| 4 | Filter baris kotor (Age non-numerik) |
| 5 | Cast tipe data eksplisit |
| 6 | Drop duplikat |

In [6]:
# ── Baca dari Bronze ──────────────────────────────────────────────────────
df_silver = spark.read.parquet(BRONZE_PATH)

# Drop kolom metadata Bronze sebelum proses
df_silver = df_silver.drop("_ingested_at", "_source_file", "_layer")

print("📥 Bronze dibaca:", df_silver.count(), "baris")

📥 Bronze dibaca: 1476 baris


In [7]:
# ── Step 1: Fix BOM character (ï»¿) di nama kolom ─────────────────────────
df_silver = df_silver.toDF(*[c.replace("\ufeff", "").replace("ï»¿", "") for c in df_silver.columns])
print("✅ Step 1 — BOM fix selesai")
print("   Kolom:", df_silver.columns[:5], "...")

✅ Step 1 — BOM fix selesai
   Kolom: ['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department'] ...


In [8]:
# ── Step 2: Rename Attrition → left_company, encode Yes/No → 1/0 ──────────
df_silver = df_silver.withColumnRenamed("Attrition", "left_company")
df_silver = df_silver.withColumn(
    "left_company",
    F.when(F.col("left_company") == "Yes", 1).otherwise(0).cast(IntegerType())
)

# Encode OverTime juga
df_silver = df_silver.withColumn(
    "OverTime",
    F.when(F.col("OverTime") == "Yes", 1).otherwise(0).cast(IntegerType())
)

attrition_dist = df_silver.groupBy("left_company").count().collect()
print("✅ Step 2 — Encode selesai. Distribusi left_company:", {r['left_company']: r['count'] for r in attrition_dist})

✅ Step 2 — Encode selesai. Distribusi left_company: {1: 237, 0: 1239}


In [9]:
# ── Step 3: Drop kolom tidak informatif ───────────────────────────────────
drop_cols = ["EmployeeCount", "StandardHours", "Over18", "EmployeeNumber"]
df_silver = df_silver.drop(*drop_cols)
print(f"✅ Step 3 — Drop kolom: {drop_cols}")
print(f"   Sisa kolom: {len(df_silver.columns)}")

✅ Step 3 — Drop kolom: ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber']
   Sisa kolom: 31


In [10]:
# ── Step 4: Filter baris kotor (Age harus numerik) ────────────────────────
before = df_silver.count()
df_silver = df_silver.filter(F.col("Age").cast("int").isNotNull())
after = df_silver.count()
print(f"✅ Step 4 — Filter baris kotor: {before - after} baris dibuang (tersisa {after})")

✅ Step 4 — Filter baris kotor: 6 baris dibuang (tersisa 1470)


In [11]:
# ── Step 5: Cast tipe data eksplisit ──────────────────────────────────────
int_cols = [
    "Age", "MonthlyIncome", "JobSatisfaction",
    "PerformanceRating", "WorkLifeBalance", "YearsAtCompany",
    "DailyRate", "DistanceFromHome", "Education",
    "EnvironmentSatisfaction", "HourlyRate", "JobInvolvement",
    "JobLevel", "MonthlyRate", "NumCompaniesWorked",
    "PercentSalaryHike", "RelationshipSatisfaction",
    "StockOptionLevel", "TotalWorkingYears",
    "TrainingTimesLastYear", "YearsInCurrentRole",
    "YearsSinceLastPromotion", "YearsWithCurrManager"
]

for c in int_cols:
    if c in df_silver.columns:
        df_silver = df_silver.withColumn(c, F.col(c).cast(IntegerType()))

print("✅ Step 5 — Cast selesai")

✅ Step 5 — Cast selesai


In [12]:
# ── Step 6: Drop duplikat ─────────────────────────────────────────────────
before = df_silver.count()
df_silver = df_silver.dropDuplicates()
after = df_silver.count()
print(f"✅ Step 6 — Duplikat dihapus: {before - after} (tersisa {after})")

# Tambah metadata Silver
df_silver = df_silver \
    .withColumn("_layer",        F.lit("silver")) \
    .withColumn("_processed_at", F.lit(datetime.utcnow().isoformat()))

print(f"\n🥈 Silver final — {df_silver.count()} baris, {len(df_silver.columns)} kolom")
df_silver.printSchema()

✅ Step 6 — Duplikat dihapus: 0 (tersisa 1470)

🥈 Silver final — 1470 baris, 33 kolom
root
 |-- Age: integer (nullable = true)
 |-- left_company: integer (nullable = false)
 |-- BusinessTravel: string (nullable = true)
 |-- DailyRate: integer (nullable = true)
 |-- Department: string (nullable = true)
 |-- DistanceFromHome: integer (nullable = true)
 |-- Education: integer (nullable = true)
 |-- EducationField: string (nullable = true)
 |-- EnvironmentSatisfaction: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- HourlyRate: integer (nullable = true)
 |-- JobInvolvement: integer (nullable = true)
 |-- JobLevel: integer (nullable = true)
 |-- JobRole: string (nullable = true)
 |-- JobSatisfaction: integer (nullable = true)
 |-- MaritalStatus: string (nullable = true)
 |-- MonthlyIncome: integer (nullable = true)
 |-- MonthlyRate: integer (nullable = true)
 |-- NumCompaniesWorked: integer (nullable = true)
 |-- OverTime: integer (nullable = false)
 |-- PercentSalaryHik

In [13]:
# ── Tulis Silver ke MinIO, partisi per Department ─────────────────────────
(
    df_silver
    .write
    .mode("overwrite")
    .partitionBy("Department")
    .parquet(SILVER_PATH)
)

print(f"✅ Silver → {SILVER_PATH}")

# Preview sample
df_silver.select(
    "Age", "Department", "JobRole", "Gender", "OverTime",
    "MonthlyIncome", "JobSatisfaction", "WorkLifeBalance",
    "YearsAtCompany", "left_company"
).show(5)

✅ Silver → s3a://datalake/processed/silver/hr-attrition/
+---+--------------------+--------------------+------+--------+-------------+---------------+---------------+--------------+------------+
|Age|          Department|             JobRole|Gender|OverTime|MonthlyIncome|JobSatisfaction|WorkLifeBalance|YearsAtCompany|left_company|
+---+--------------------+--------------------+------+--------+-------------+---------------+---------------+--------------+------------+
| 49|Research & Develo...|  Research Scientist|  Male|       0|         5130|              2|              3|            10|           0|
| 34|Research & Develo...|Laboratory Techni...|  Male|       0|         2661|              4|              3|             2|           0|
| 22|Research & Develo...|Laboratory Techni...|  Male|       1|         2935|              4|              2|             1|           0|
| 53|               Sales|             Manager|Female|       0|        15427|              4|              3|      

---
## 🥇 GOLD — ML-Ready Features
> Baca Silver → StringIndexer → VectorAssembler → MinMaxScaler
> Output identik dengan `final_dataset.csv` dari notebook KBA.

| Kolom Output | Keterangan |
|---|---|
| `scaled_features` | Vector: Age, MonthlyIncome, JobSatisfaction, PerformanceRating, WorkLifeBalance, YearsAtCompany (dinormalisasi 0–1) |
| `Department_index` | Label encoded |
| `JobRole_index` | Label encoded |
| `Gender_index` | Label encoded |
| `OverTime_index` | Label encoded |
| `left_company` | Target (0=stay, 1=keluar) |

In [14]:
from pyspark.ml.feature import StringIndexer, VectorAssembler, MinMaxScaler
from pyspark.ml import Pipeline

# ── Baca Silver (tanpa kolom metadata) ───────────────────────────────────
df_gold = (
    spark.read
    .parquet(SILVER_PATH)
    .drop("_layer", "_processed_at")
)
print("📥 Silver dibaca:", df_gold.count(), "baris")

📥 Silver dibaca: 1470 baris


In [15]:
# ── Step 1: StringIndexer — encode kolom kategorik ─────────────────────────
# Sesuai notebook KBA: Department, JobRole, Gender, MaritalStatus, OverTime
# (MaritalStatus tidak masuk final_dataset, tapi tetap diindex)
categorical_cols = ["Department", "JobRole", "Gender", "MaritalStatus", "OverTime"]

indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_index", handleInvalid="keep")
    for c in categorical_cols
]

for indexer in indexers:
    df_gold = indexer.fit(df_gold).transform(df_gold)

print("✅ StringIndexer selesai")
df_gold.select("Department", "Department_index", "JobRole", "JobRole_index").show(3)

✅ StringIndexer selesai
+--------------------+----------------+--------------------+-------------+
|          Department|Department_index|             JobRole|JobRole_index|
+--------------------+----------------+--------------------+-------------+
|Research & Develo...|             0.0|  Research Scientist|          1.0|
|Research & Develo...|             0.0|Laboratory Techni...|          2.0|
|Research & Develo...|             0.0|Laboratory Techni...|          2.0|
+--------------------+----------------+--------------------+-------------+
only showing top 3 rows



In [16]:
# ── Step 2: VectorAssembler — gabung fitur numerik ────────────────────────
# Sesuai notebook KBA
numeric_cols = [
    "Age", "MonthlyIncome", "JobSatisfaction",
    "PerformanceRating", "WorkLifeBalance", "YearsAtCompany"
]

assembler = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="num_features",
    handleInvalid="keep"
)
df_gold = assembler.transform(df_gold)

print("✅ VectorAssembler selesai")
df_gold.select("num_features").show(3, truncate=False)

✅ VectorAssembler selesai
+------------------------------+
|num_features                  |
+------------------------------+
|[49.0,5130.0,2.0,4.0,3.0,10.0]|
|[34.0,2661.0,4.0,3.0,3.0,2.0] |
|[22.0,2935.0,4.0,3.0,2.0,1.0] |
+------------------------------+
only showing top 3 rows



In [17]:
# ── Step 3: MinMaxScaler — normalisasi 0–1 ────────────────────────────────
scaler = MinMaxScaler(inputCol="num_features", outputCol="scaled_features")
df_gold = scaler.fit(df_gold).transform(df_gold)

print("✅ MinMaxScaler selesai")
df_gold.select("scaled_features").show(3, truncate=False)

✅ MinMaxScaler selesai
+--------------------------------------------------------------------------------------+
|scaled_features                                                                       |
+--------------------------------------------------------------------------------------+
|[0.738095238095238,0.21700895208004214,0.3333333333333333,1.0,0.6666666666666666,0.25]|
|[0.38095238095238093,0.08699315429173249,1.0,0.0,0.6666666666666666,0.05]             |
|[0.09523809523809523,0.1014218009478673,1.0,0.0,0.3333333333333333,0.025]             |
+--------------------------------------------------------------------------------------+
only showing top 3 rows



In [18]:
# ── Step 4: Pilih kolom final (identik dengan final_dataset.csv) ──────────
df_final = df_gold.select(
    "scaled_features",
    "Department_index",
    "JobRole_index",
    "Gender_index",
    "OverTime_index",
    "left_company"
)

print(f"🥇 Gold final — {df_final.count()} baris, {len(df_final.columns)} kolom")
df_final.show(5, truncate=False)

🥇 Gold final — 1470 baris, 6 kolom
+---------------------------------------------------------------------------------------+----------------+-------------+------------+--------------+------------+
|scaled_features                                                                        |Department_index|JobRole_index|Gender_index|OverTime_index|left_company|
+---------------------------------------------------------------------------------------+----------------+-------------+------------+--------------+------------+
|[0.738095238095238,0.21700895208004214,0.3333333333333333,1.0,0.6666666666666666,0.25] |0.0             |1.0          |0.0         |0.0           |0           |
|[0.38095238095238093,0.08699315429173249,1.0,0.0,0.6666666666666666,0.05]              |0.0             |2.0          |0.0         |0.0           |0           |
|[0.09523809523809523,0.1014218009478673,1.0,0.0,0.3333333333333333,0.025]              |0.0             |2.0          |0.0         |1.0           |0      

In [20]:
# ── Tulis Gold ke MinIO (Parquet) ─────────────────────────────────────────
GOLD_ML_PATH = f"{GOLD_PATH}ml_features/"
(
    df_final
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(GOLD_ML_PATH)
)
print(f"✅ Gold → {GOLD_ML_PATH}")

# ── Simpan CSV untuk verifikasi (konversi vector ke string dulu) ──────────
from pyspark.ml.functions import vector_to_array

GOLD_CSV_PATH = f"{GOLD_PATH}ml_features_csv/"
(
    df_final
    .withColumn("scaled_features", vector_to_array("scaled_features").cast("string"))
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(GOLD_CSV_PATH)
)
print(f"✅ Gold CSV → {GOLD_CSV_PATH}")

✅ Gold → s3a://datalake/processed/gold/hr-attrition/ml_features/
✅ Gold CSV → s3a://datalake/processed/gold/hr-attrition/ml_features_csv/


---
## ✅ Verifikasi Akhir — Semua Layer

In [21]:
print("=" * 65)
print("📋 RINGKASAN MEDALLION PIPELINE — HR ATTRITION")
print("=" * 65)
print(f"   Bucket : datalake")
print(f"   Layout :")
print(f"     raw/hr-attrition/                    ← 🥉 Bronze")
print(f"     processed/silver/hr-attrition/       ← 🥈 Silver")
print(f"     processed/gold/hr-attrition/         ← 🥇 Gold")
print()

layers = [
    ("raw/hr-attrition/",                  "🥉 Bronze"),
    ("processed/silver/hr-attrition/",     "🥈 Silver"),
    ("processed/gold/hr-attrition/ml_features/", "🥇 Gold"),
]

for prefix, label in layers:
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    files = [f for f in resp.get("Contents", []) if not f["Key"].endswith("/")]
    total_bytes = sum(f["Size"] for f in files)
    print(f"{label} ({prefix})")
    print(f"   {len(files)} file(s) | {total_bytes:,} bytes")

print("\n" + "-" * 65)
print("📊 Row Counts:")
bronze_df = spark.read.parquet(BRONZE_PATH)
silver_df = spark.read.parquet(SILVER_PATH)
gold_df   = spark.read.parquet(f"{GOLD_PATH}ml_features/")
print(f"   🥉 Bronze  : {bronze_df.count():,} baris")
print(f"   🥈 Silver  : {silver_df.count():,} baris")
print(f"   🥇 Gold    : {gold_df.count():,} baris")
print()
print("🎯 Distribusi Target (left_company):")
gold_df.groupBy("left_company").count().orderBy("left_company").show()
print("=" * 65)
print("🏁 Pipeline selesai!")

📋 RINGKASAN MEDALLION PIPELINE — HR ATTRITION
   Bucket : datalake
   Layout :
     raw/hr-attrition/                    ← 🥉 Bronze
     processed/silver/hr-attrition/       ← 🥈 Silver
     processed/gold/hr-attrition/         ← 🥇 Gold

🥉 Bronze (raw/hr-attrition/)
   2 file(s) | 54,017 bytes
🥈 Silver (processed/silver/hr-attrition/)
   4 file(s) | 67,261 bytes
🥇 Gold (processed/gold/hr-attrition/ml_features/)
   2 file(s) | 27,382 bytes

-----------------------------------------------------------------
📊 Row Counts:
   🥉 Bronze  : 1,476 baris
   🥈 Silver  : 1,470 baris
   🥇 Gold    : 1,470 baris

🎯 Distribusi Target (left_company):
+------------+-----+
|left_company|count|
+------------+-----+
|           0| 1233|
|           1|  237|
+------------+-----+

🏁 Pipeline selesai!


In [22]:
spark.stop()
print("🛑 SparkSession dihentikan")

🛑 SparkSession dihentikan
